<div style="display:flow-root;">
<img src="https://raw.githubusercontent.com/ColinConwell/DL4HL-Workshop/main/assets/visna-logo-rounded.svg" alt="VISNA Lab" width="125" height="125" align="left" style="float:left;border-radius:26px;margin:0 24px 16px 0;object-fit:contain;">
<h1 style="margin:0 0 12px;padding:0;">Deep Learning for Human Learners</h1>
<p style="margin:0;">Part 2: Nonlinearity, Disentanglement, &amp; Pretrained Representations</p>
<br clear="all">
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ColinConwell/DL4HL-Workshop/blob/main/DL4HL-Tutorial.ipynb)

In [ ]:
# @title In Colab? Click Play
import importlib.util, subprocess, sys

_REQ = [
    ("open_clip", "open-clip-torch"),
    ("horama", "horama"),
    ("overcomplete", "overcomplete"),
    ("skorch", "skorch"),
    ("lightning", "lightning"),
    ("torchinfo", "torchinfo"),
    ("torchlens", "torchlens"),
    ("huggingface_hub", "huggingface_hub"),
    ("pyarrow", "pyarrow"),
    ("seaborn", "seaborn"),
]
missing = [pkg for mod, pkg in _REQ if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Download this notebook's reviewed helper version (Python, JavaScript, and CSS).
import hashlib, io, tempfile, zipfile, os
from urllib.request import urlopen
os.environ["DL4HL_STANDALONE"] = "1"
for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_thread_variable] = os.environ.get("DL4HL_THREADS", "2")
_helper_sha256 = '8c389497bde296269b59324b1320a5e7134bc6f19b78037caa4635e5b8f96969'
_helper_url = (
    "https://raw.githubusercontent.com/ColinConwell/DL4HL-Workshop/main/"
    f"assets/helpers/dl4hl-{_helper_sha256}.zip"
)
with urlopen(_helper_url, timeout=60) as _response:
    _helper_zip = _response.read()
if hashlib.sha256(_helper_zip).hexdigest() != _helper_sha256:
    raise RuntimeError("Helper download failed its SHA-256 check; nothing was loaded.")
_helper_dir = tempfile.mkdtemp(prefix="dl4hl-helper-")
with zipfile.ZipFile(io.BytesIO(_helper_zip)) as _archive:
    _archive.extractall(_helper_dir)
sys.path.insert(0, _helper_dir)
for _name in list(sys.modules):
    if _name == "dl4hl" or _name.startswith("dl4hl."):
        del sys.modules[_name]
import sklearn
from packaging.version import Version
if Version(sklearn.__version__) < Version("1.6"):
    raise RuntimeError("This notebook needs scikit-learn >= 1.6. Upgrade it and restart the runtime.")
import dl4hl
from IPython.display import HTML, display
display(HTML(dl4hl.theming.theme_html()))


In the [introductory workshop](https://colinconwell.github.io/ML4HL-Workshop/), we built learning machines out of single geometric moves: tilt a flat line, draw a boundary, group nearby dots, or project high-dimensional clouds onto flat shadows. What happens when the phenomena we study refuse to sit still along flat axes — when two pixels, two words, or two clinical risk factors reverse each other's meaning depending on context, or when raw photographs, spectrograms, and sentences arrive without tidy spreadsheet columns? Deep learning answers by stacking many simple coordinate warps end to end, letting the machine construct its own intermediate features on the way to a prediction. In this tutorial, we poke directly at the mechanics that make depth work, run modern PyTorch and Hugging Face workflows across tables, images, and text, open the lid with mechanistic interpretability tools, and turn open-source model zoos into controlled experiments for the human sciences.

**How to use this page.** Every figure with a play button, slider, or 3D sphere is live: drag handles, rotate latent spaces, toggle layers, and read the status strip beneath each canvas. The code blocks run real [PyTorch](https://pytorch.org), [Hugging Face](https://huggingface.co), [OpenCLIP](https://github.com/mlfoundations/open_clip), and [scikit-learn](https://scikit-learn.org) analyses on open empirical datasets. Prefer to run and modify the code yourself? The [generated notebook opens directly in Google Colab](https://colab.research.google.com/github/ColinConwell/DL4HL-Workshop/blob/main/DL4HL-Tutorial.ipynb) with all interactive figures intact. [Download the source and notebook](https://github.com/ColinConwell/DL4HL-Workshop).

<details>
<summary>Abbreviations Used in the Figures</summary>

**MLP** means multilayer perceptron (a feedforward stack of weighted sums and nonlinearities); **ReLU** means rectified linear unit ($\max(0, z)$, a piecewise-linear hinge); and **GELU** means Gaussian error linear unit (a smooth probabilistic bend used in modern transformers). **CNN** means convolutional neural network, **ViT** means Vision Transformer, and **LLM** means large language model. **LoRA** means low-rank adaptation (updating small adapter matrices while keeping pretrained weights frozen). **SAE** means sparse autoencoder (an overcomplete dictionary with a sparsity constraint), **MACO** means magnitude-constrained optimization (synthesizing feature visualizations via phase optimization in Fourier space), **SRP** means sparse random projection (fast distance-preserving compression), and **RDM** means representational dissimilarity matrix.

</details>

- **Nonlinearity &amp; Disentanglement**: bend curves, fold XOR, &amp; untangle manifolds
- **Tabular Deep Learning**: PyTorch engines inside scikit-learn &amp; Lightning
- **Vision &amp; Feature Attribution**: transfer learning, graph tracing, &amp; MACO
- **Language Models &amp; SAEs**: token trajectories, LoRA, &amp; sparse dictionaries
- **Multimodal &amp; Relative Spaces**: zero-shot CLIP &amp; natural-language anchor queries
- **Encoding, Decoding, &amp; Model Zoos**: DeepDive hierarchies &amp; opportunistic experiments

In [ ]:
# @title Setup: imports, thread bounds, and the dl4hl helper
import os
import sys

N_JOBS = int(os.environ.get("DL4HL_N_JOBS", "2"))
N_THREADS = int(os.environ.get("DL4HL_THREADS", "2"))
for _thread_variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_thread_variable] = str(N_THREADS)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import dl4hl  # Installed locally, or bundled by the generated Colab setup.
from dl4hl import show
from dl4hl.samples import widget_sample

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from threadpoolctl import threadpool_limits

torch.set_num_threads(N_THREADS)
_thread_limit = threadpool_limits(limits=N_THREADS)

dl4hl.reset_assets()
dl4hl.set_style()
dl4hl.enable_theme_plots()
C = dl4hl.COLORS
rng = np.random.default_rng(2026)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Research Domains in This Tutorial

Each section pairs interactive simulations with executable PyTorch analyses on public datasets spanning our audience's research areas:

- **Anthropology: household surveys &amp; visual culture**: World Bank Living Standards Measurement Study (LSMS) commune-grouped household expenditure; Gapminder Dollar Street domestic photography projected onto natural-language anchor queries.
- **Developmental &amp; clinical psychology**: Stanford Wordbank MacArthur–Bates CDI infant vocabulary trajectories; Infant Health and Development Program (IHDP) perinatal risk and childhood reading achievement.
- **Music therapy, affect, &amp; aesthetics**: Cowen et al. 13-dimensional listener emotion ratings across 1,841 musical excerpts; Kurdi et al. Open Affective Standardized Image Set (OASIS) beauty, valence, and arousal judgments.
- **Perception, psycholinguistics, &amp; pragmatics**: Blender ShapeCentral 3D objects with surface-texture foils; Floyd et al. parametric Kiki–Bouba sound-symbolism shapes; 28,619 straight vs. sarcastic news headlines.

# Nonlinearity and Disentanglement

Before loading million-parameter vision or language backbones, we need three geometric intuitions that explain why stacking layers accomplishes anything a single regression cannot: **nonlinear hinges**, **interaction folding (XOR)**, and **progressive disentanglement**.

## What Is a Nonlinearity? Spline Regression as Learnable Hinges

Suppose you stack ten linear transformations in a row, feeding the weighted sum of one layer straight into the weighted sum of the next. Algebra plays a rude trick: a linear function of a linear function is still just a single linear function. Ten stacked matrices without anything nonlinear in between collapse into one flat line or hyperplane, buying zero expressive power in exchange for ten times the bookkeeping.

To bend a prediction curve, a network inserts a simple elementwise **activation function** after each weighted sum. In spline regression, we bend a line by placing **knots** along the input axis where the slope is allowed to change. A **rectified linear unit (ReLU)** is a learnable spline knot: it outputs zero below its threshold and rises linearly above it. Smooth relatives like **GELU** (standard in transformers) and **Tanh** round off that corner while serving the same geometric role: letting weighted combinations of local bends approximate curved trajectories.

Switch the **Activation** toggle below to **Linear** first — notice how even twelve hidden units collapse onto a single straight line — then switch to **ReLU** or **GELU** and press **play** (or drag the orange knot handles along the bottom axis):

In [ ]:
# @title Show the interactive figure code
show("activation_splines")

We see the same geometry on empirical developmental data when modeling early productive vocabulary growth across 8 to 30 months in the Stanford Wordbank repository (`load_wordbank`):

In [ ]:
# @title Show the interactive figure code
show("activation_splines", data=widget_sample("activation_splines"), units=6, activation="gelu")

In Python, fitting a one-hidden-layer PyTorch network with ReLU activations learns adaptive hinge positions directly by gradient descent, playing the adaptive counterpart to fixed-knot `SplineTransformer` regression:

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler
import torch.nn as nn

wb = dl4hl.load_wordbank()
ages = wb[["age"]].to_numpy(dtype=np.float32)
vocab = wb["vocab_share"].to_numpy(dtype=np.float32)

# Compare a linear baseline, a fixed-knot cubic spline, and a small PyTorch ReLU network.
cv5 = KFold(5, shuffle=True, random_state=0)
linear_r2 = cross_val_score(make_pipeline(StandardScaler(), RidgeCV()), ages, vocab, cv=cv5).mean()
spline_r2 = cross_val_score(
    make_pipeline(SplineTransformer(n_knots=6, degree=3), RidgeCV()), ages, vocab, cv=cv5
).mean()

torch.manual_seed(2026)
age_z = (ages - ages.mean()) / ages.std()
relu_curve = nn.Sequential(nn.Linear(1, 12), nn.ReLU(), nn.Linear(12, 1))
opt = torch.optim.Adam(relu_curve.parameters(), lr=0.03)
x_t, y_t = torch.from_numpy(age_z), torch.from_numpy(vocab[:, None])
for _ in range(250):
    opt.zero_grad()
    nn.functional.mse_loss(relu_curve(x_t), y_t).backward()
    opt.step()

age_grid = np.linspace(ages.min(), ages.max(), 120, dtype=np.float32)[:, None]
with torch.no_grad():
    relu_pred = relu_curve(torch.from_numpy((age_grid - ages.mean()) / ages.std())).squeeze().numpy()
spline_fit = make_pipeline(SplineTransformer(n_knots=6, degree=3), RidgeCV()).fit(ages, vocab).predict(age_grid)
print(f"5-fold R² — straight line: {linear_r2:.3f} | 6-knot spline: {spline_r2:.3f}")

In [ ]:
# @title Show the figure code
fig, ax = plt.subplots(figsize=(7.6, 3.4))
sub_i = rng.choice(len(ages), size=min(600, len(ages)), replace=False)
ax.scatter(ages[sub_i, 0], vocab[sub_i], s=12, alpha=0.25, color=C["muted"], label="Wordbank administrations")
ax.plot(age_grid[:, 0], spline_fit, lw=2.3, color=C["green"], label="Cubic spline + RidgeCV")
ax.plot(age_grid[:, 0], relu_pred, "--", lw=2.3, color=C["blue"], label="PyTorch 12-unit ReLU network")
ax.set(xlabel="child age (months)", ylabel="proportion of checklist produced", title="Fixed spline knots vs. learned ReLU hinges on infant vocabulary")
ax.legend()
plt.tight_layout()

> **Takeaway #1**
>
> Stacking linear layers without nonlinear activations collapses to a single linear transformation; nonlinear hinges let a network approximate curved functions by combining local bends.

## Why Do We Need Nonlinearity? The XOR Problem

Curvature in one predictor is convenient, yet the deeper reason neural networks need nonlinearities appears as soon as **two or more inputs interact**. Consider the classic **exclusive-OR (XOR)** pattern, which Minsky and Papert (1969) famously used to show the limits of single-layer perceptrons: Category B occurs when $x_1$ is high *or* $x_2$ is high, **except when both are high together**. Clinical and behavioral phenomena take this form routinely: physiological arousal paired with positive appraisal feels invigorating, low arousal paired with negative appraisal feels flat, whereas high arousal paired with negative appraisal signals acute distress. Neither axis has a consistent main effect on its own; each coordinate's meaning flips sign depending on the other.

Try **0 hidden (linear)** in Figure 2 below and press **play**: a single straight boundary is mathematically trapped because opposite corners of a square cannot sit on the same side of one line without dragging at least one mismatched corner along. Switch to **1 hidden layer**, press **play**, and drag the **Fold morph** slider from 0% to 100% to watch what the hidden ReLU creases do to the space itself:

In [ ]:
# @title Show the interactive figure code
show("xor_fold")

Instead of drawing a wiggly rule in the original coordinates, the hidden layer **folds the sheet of paper** along its ReLU creases until the two blue corners land on top of each other and a single straight cut in the final layer separates the classes cleanly.

> **Takeaway #2**
>
> A single linear boundary cannot separate interaction patterns like XOR, where an input's meaning reverses depending on context; hidden nonlinear layers re-represent the inputs so a linear boundary at the end suffices.

## Deep Learning as Disentanglement: Untangling Christmas Lights

Why stack *many* layers rather than cramming thousands of hinges into a single wide layer? Picture pulling two strings of Christmas lights — one blue strand, one orange strand — out of a closet box twisted tightly around each other. Try separating them in a single decisive chop with a flat ruler (a linear classifier in pixel or sensor space) and you slice right through both cords. Try yanking every loop at once and the knot cinches tighter. Instead, you loosen one outer loop, rotate a twist slightly, slide a strand free, and repeat — **pulling at one knot at a time**.

Each hidden block in a deep residual network performs that kind of gentle coordinate warp: shift features slightly (`x + f(x)`), crease along a few ReLU hinges, and hand the partially loosened geometry to the next block. Press **play** or scrub **layer depth** from `L0` to `L4` in Figure 3 below to watch two intertwined spiral manifolds untangle across four learned residual layers while a linear probe's accuracy climbs from near-chance to complete separation:

In [ ]:
# @title Show the interactive figure code
show("manifold_untangle")

Real sensory inputs behave like high-dimensional versions of those tangled cords. Two photographs of the same person under harsh side-lighting versus soft shade can sit farther apart in raw pixel values than photographs of two strangers under matching light; two recordings of a lullaby sung in different vocal registers can differ across almost every raw frequency bin. Depth lets a network peel away nuisance variation — pose, illumination, pitch register, phrasing — step by step until task-relevant structure lies flat enough for a simple linear readout at the top.

> **Takeaway #3**
>
> Deep networks untangle complex data manifolds progressively across layers, applying a sequence of simple warps and folds until target categories or quantities become linearly readable.

# Deep Learning on Structured Tables

Before moving to images and text, how do neural networks behave on the bread-and-butter data format of the social, behavioral, and clinical sciences: **structured spreadsheets** of survey indicators, longitudinal scores, and rating scales? Under the hood, every training step runs two passes across the network graph: a **forward pass** that multiplies inputs by weights and passes sums through activations to form a prediction, and a **backward pass (backpropagation)** that applies the chain rule from the prediction miss back through every layer so each weight learns how strongly it contributed to the error.

Press **play** or **step** in Figure 4 below, toggle between **Weight sign** and **Backprop gradient**, and slide the standardized predictors to watch signals travel left-to-right and credit travel right-to-left:

In [ ]:
# @title Show the interactive figure code
show("mlp_backprop")

## Scikit-Learn Ergonomics with a PyTorch Engine (`skorch`)

Writing raw PyTorch training loops just to cross-validate a tabular regression inside a `GroupKFold` pipeline would quickly grow tedious — and worse, invites subtle data-leakage bugs if standardization happens outside the fold split. Libraries like [`skorch`](https://skorch.readthedocs.io/) wrap any `torch.nn.Module` in the exact `.fit(X, y)` and `.predict(X)` interface of `scikit-learn`, letting us drop a PyTorch network straight into `make_pipeline(StandardScaler(), ...)` alongside `RidgeCV` and `HistGradientBoostingRegressor`.

Below, we define a compact residual tabular MLP in PyTorch and benchmark it on two grouped empirical tables from [Part 1](https://colinconwell.github.io/ML4HL-Workshop/): predicting household log expenditure across 194 communes in the **World Bank Living Standards Measurement Study (`load_household_survey`)**, and predicting age-8 reading achievement across clinical sites in the **Infant Health and Development Program (`load_perinatal_dev`)**.

<details>
<summary>Measurement and Grouping Notes</summary>

In the LSMS household table, folds hold out entire **communes** (`GroupKFold`) so commune-level health and infrastructure summaries never leak between training and test households; contemporaneous medical spending is excluded from predictors when modeling total expenditure to avoid mechanical part-whole overlap. In the IHDP cohort, folds hold out **clinical sites**, using the original neonatal and demographic covariates directly rather than any full-sample composite.

</details>

In [ ]:
from skorch import NeuralNetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

class ResidualTabularMLP(nn.Module):
    def __init__(self, n_features: int, hidden: int = 48, dropout: float = 0.1):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(n_features, hidden), nn.GELU())
        self.block = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, hidden)
        )
        self.head = nn.Linear(hidden, 1)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        h = self.stem(X)
        return self.head(h + self.block(h))

def make_torch_regressor(n_features: int) -> NeuralNetRegressor:
    torch.manual_seed(2026)
    return NeuralNetRegressor(
        ResidualTabularMLP,
        module__n_features=n_features,
        max_epochs=35,
        lr=3e-3,
        batch_size=64,
        optimizer=torch.optim.AdamW,
        optimizer__weight_decay=1e-2,
        train_split=False,
        verbose=0,
        device=DEVICE,
    )

# 1. Anthropology: LSMS household log total expenditure held out by commune.
lsms = dl4hl.load_household_survey()
lsms_cols = [
    "urban", "farm", "hhsize", "head_age", "head_female", "educ_years",
    "commune_illness_rate", "commune_insurance_rate", "commune_pharmacy_visits",
]
X_lsms = lsms[lsms_cols].to_numpy(dtype=np.float32)
y_lsms = lsms[["log_total_exp"]].to_numpy(dtype=np.float32)

# 2. Developmental psychology: IHDP age-8 Woodcock-Johnson reading held out by clinical site.
ihdp_cols = [
    "birthweight_g", "gestational_age_wk", "head_circumference_cm", "neonatal_health",
    "apgar5", "prenatal_care_trimester", "prenatal_cigs_per_day", "maternal_age",
    "maternal_ed_level", "intervention",
]
ihdp = dl4hl.load_perinatal_dev().dropna(subset=ihdp_cols + ["reading_96m"]).reset_index(drop=True)
X_ihdp = ihdp[ihdp_cols].to_numpy(dtype=np.float32)
y_ihdp = ihdp[["reading_96m"]].to_numpy(dtype=np.float32)

Now evaluate `RidgeCV`, `HistGradientBoostingRegressor`, and our `skorch`-wrapped PyTorch `ResidualTabularMLP` under identical `GroupKFold` splits:

In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GroupKFold

bench_rows = []
tasks = [
    ("LSMS expenditure (by commune)", X_lsms, y_lsms, lsms["commune"].to_numpy()),
    ("IHDP age-8 reading (by site)", X_ihdp, y_ihdp, ihdp["site"].to_numpy()),
]
for task_name, X_mat, y_col, groups in tasks:
    gkf = GroupKFold(n_splits=5)
    torch_pipe = TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), make_torch_regressor(X_mat.shape[1])),
        transformer=StandardScaler(),
    )
    estimators = {
        "Linear (RidgeCV)": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 15))),
        "Tree ensemble (HistGBM)": HistGradientBoostingRegressor(max_depth=4, random_state=0),
        "PyTorch MLP (skorch)": torch_pipe,
    }
    for model_label, est in estimators.items():
        y_input = y_col if "PyTorch" in model_label else y_col.ravel()
        scores = cross_val_score(est, X_mat, y_input, groups=groups, cv=gkf, scoring="r2")
        bench_rows.append({"dataset": task_name, "model": model_label, "mean_r2": scores.mean(), "sd_r2": scores.std()})

tabular_bench = pd.DataFrame(bench_rows)
tabular_bench.round(3)

Notice the realistic pattern in the table above: on compact tabular spreadsheets with dozens to a few thousand rows, regularized linear models and gradient-boosted decision trees match or slightly edge out a neural network while requiring minimal hyperparameter tuning. Where PyTorch earns a distinct advantage on structured data is **multi-output representation learning** and **end-to-end differentiable objectives** — predicting an entire profile of correlated outcomes through a shared bottleneck, or fusing tabular covariates with images and text.

## Multi-Target Scoring with PyTorch Lightning

To see where a PyTorch workflow shines on structured ratings, we use [`lightning`](https://lightning.ai/docs/pytorch/stable/) (`LightningModule` + `Trainer`) paired with `torchmetrics` on [Cowen et al.'s (2020)](https://doi.org/10.1073/pnas.1910704117) music-emotion dataset (`load_music_emotions`, $1,841$ musical excerpts). Given listener ratings on six non-target appraisal dimensions (such as *amusing*, *dreamy*, *eerie/mysterious*, and *triumphant/heroic*), a single multi-head network learns a shared latent representation that predicts four clinically relevant affective regulatory targets (*calm/relaxing/serene*, *compassionate/sympathetic*, *energizing/pump-up*, and *anxious/tense*) simultaneously on whatever accelerator (`CPU`, `MPS`, or `CUDA`) is available:

In [ ]:
import logging
import lightning as L
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics import R2Score
from sklearn.model_selection import train_test_split

logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)

music = dl4hl.load_music_emotions()
emo_df = music["df"]
targets_4 = ["calm/relaxing/serene", "compassionate/sympathetic", "energizing/pump-up", "anxious/tense"]
inputs_other = [c for c in music["core13"] if c not in targets_4]

X_tr, X_te, Y_tr, Y_te = train_test_split(
    emo_df[inputs_other].to_numpy(dtype=np.float32),
    emo_df[targets_4].to_numpy(dtype=np.float32),
    test_size=0.25,
    random_state=2026,
)
x_scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = x_scaler.transform(X_tr).astype(np.float32), x_scaler.transform(X_te).astype(np.float32)

class MultiAffectModule(L.LightningModule):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32), nn.GELU(), nn.Linear(32, 16), nn.GELU(), nn.Linear(16, out_dim)
        )
        self.val_r2 = R2Score(multioutput="raw_values")
        self.latest_r2 = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        return nn.functional.mse_loss(self(x), y)

    def validation_step(self, batch, batch_idx):
        x, y = batch
        self.val_r2.update(self(x), y)

    def on_validation_epoch_end(self):
        self.latest_r2 = self.val_r2.compute().detach().cpu().numpy()
        self.val_r2.reset()

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=4e-3, weight_decay=1e-2)

L.seed_everything(2026, workers=True)
lit_model = MultiAffectModule(X_tr_s.shape[1], len(targets_4))
trainer = L.Trainer(
    max_epochs=30, accelerator="auto", devices=1, logger=False,
    enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False,
)
trainer.fit(
    lit_model,
    DataLoader(TensorDataset(torch.from_numpy(X_tr_s), torch.from_numpy(Y_tr)), batch_size=64, shuffle=True),
    DataLoader(TensorDataset(torch.from_numpy(X_te_s), torch.from_numpy(Y_te)), batch_size=128),
)
heldout_r2 = {t.split("/")[0]: float(v) for t, v in zip(targets_4, lit_model.latest_r2.round(3))}
print("Held-out multi-target R² per affective dimension:", heldout_r2)

In [ ]:
# @title Show the figure code
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.5), gridspec_kw={"width_ratios": [1.35, 1]})
pivot_df = tabular_bench.pivot(index="dataset", columns="model", values="mean_r2")
x_pos = np.arange(len(pivot_df.index))
bar_cols = [C["blue"], C["green"], C["orange"]]
for idx, col_name in enumerate(["Linear (RidgeCV)", "Tree ensemble (HistGBM)", "PyTorch MLP (skorch)"]):
    axes[0].bar(x_pos + (idx - 1) * 0.24, pivot_df[col_name].to_numpy(), width=0.22, color=bar_cols[idx], label=col_name)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(["LSMS expenditure\n(commune folds)", "IHDP reading\n(site folds)"])
axes[0].set(ylabel="5-fold grouped R²", title="Tabular baselines vs. skorch PyTorch MLP")
axes[0].legend(fontsize=8)

axes[1].barh(list(heldout_r2.keys()), list(heldout_r2.values()), color=C["purple"])
axes[1].set(xlim=(0, 1.0), xlabel="held-out R² (torchmetrics)", title="Lightning multi-output music affect")
plt.tight_layout()

> **Takeaway #4**
>
> Tabular neural networks pair PyTorch modules with familiar scikit-learn pipelines and grouped cross-validation, though gradient-boosted trees remain essential baselines on small structured tables.

> **Going faster**
>
> In PyTorch Lightning, changing `Trainer(accelerator="auto", precision="bf16-mixed")` moves training and `torchmetrics` evaluation onto CUDA GPUs or Apple Silicon MPS cores with half-precision tensor cores and zero changes to model code. On PyTorch 2+, `torch.compile(model)` fuses elementwise activations and linear layers into optimized kernels for long training runs.

# Deep Learning on Vision: Transfer Learning and Feature Interpretability

For decades, introductory perception courses opened with a reliable humbling fact about computers: change the camera angle, cast a shadow, or wrap a familiar shape in an unfamiliar surface texture, and pixel-based classifiers fell apart. Deep convolutional networks and Vision Transformers changed that story by learning hierarchies of spatial features — oriented edges in early layers, textures and corners in middle layers, and viewpoint-tolerant 3D parts near the output — across millions of natural images.

## Transfer Learning: Deep Performance Without Deep Time

Because open-source repositories like `torchvision` and Ross Wightman's [`timm`](https://github.com/huggingface/pytorch-image-models) distribute networks already pretrained on millions of photographs, researchers rarely train vision models from random noise. Instead, we rely on **transfer learning** in one of two regimes:

- **Fixed feature extraction (linear probing)** freezes the pretrained backbone weights and fits only a fresh final classification layer to our study's labels. For scientific benchmarking, freezing the backbone also preserves a clean inference: accuracy reflects information *already organized* by the pretrained visual system rather than features newly invented on our target set.
- **Full finetuning** allows gradients to adjust earlier backbone layers too — powerful when target images depart sharply from everyday photography (such as histopathology or satellite imagery) and sample sizes are large enough to avoid overfitting.

Explore the sample-efficiency tradeoff between **Frozen head**, **Full finetune**, and **From scratch** across dataset sizes in Figure 5 below:

In [ ]:
# @title Show the interactive figure code
show("transfer_hierarchy")

## The Shape Central Benchmark

To test 3D shape recognition without letting a network cheat via background shortcuts, we load `SimpleShapes` (`load_simple_shapes`), a dataset rendered in Blender ([ShapeCentral](https://github.com/ColinConwell/ShapeCentral)) across six geometric categories — `Cone`, `Cube`, `Cylinder`, `Sphere`, `Suzanne` (Blender's monkey mesh), and `Torus` — under randomized position, 3D rotation, lighting, and **deliberate surface-texture randomization** designed to foil pure texture-patch shortcuts:

In [ ]:
from PIL import Image

shapes_bundle = dl4hl.load_simple_shapes()
shapes_df = shapes_bundle["df"]
shape_classes = shapes_bundle["classes"]
print(f"{len(shapes_df)} rendered stimuli across {len(shape_classes)} categories: {shape_classes}")
shapes_df.groupby("val_set")["label"].value_counts().unstack()

In [ ]:
# @title Show the figure code
sample_first = [
    shapes_df.query("image_index == 1 and label == @cat")["image_path"].iloc[0]
    for cat in shape_classes
]
fig, axes = plt.subplots(1, 6, figsize=(9.6, 2.2))
for ax, path, cat in zip(axes, sample_first, shape_classes):
    ax.imshow(Image.open(path))
    ax.set_title(cat, fontsize=9)
    ax.set_axis_off()
fig.suptitle("Render #00001 across all six ShapeCentral 3D categories (note Cone vs. Sphere shadow cue)", fontsize=10)
plt.tight_layout()

Look closely at `Cone` versus `Sphere` in render `#00001` above: viewed nearly head-on under matching stippled texture, only the pointed **cast shadow** betrays the cone's 3D geometry.

## Pretrained `ResNet-18` Transfer Learning and Model Informatics

We wrap our stimulus dataframe in a standard PyTorch `Dataset` + `DataLoader`, normalize RGB channels to the ImageNet statistics expected by pretrained weights, load `resnet18`, replace its final 1,000-way linear head (`model.fc`) with a 6-way shape readout, and freeze the convolutional filters while leaving batch-normalization statistics and `model.fc` adaptable:

In [ ]:
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import torchinfo
import torchlens as tl

class ShapeDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform):
        self.paths = frame["image_path"].tolist()
        self.labels = pd.Categorical(frame["label"], categories=shape_classes).codes.astype(np.int64)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img), int(self.labels[idx])

imagenet_norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
shape_transforms = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(160, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        imagenet_norm,
    ]),
    "val": transforms.Compose([transforms.Resize((160, 160)), transforms.ToTensor(), imagenet_norm]),
}

torch.manual_seed(2026)
train_loader = DataLoader(ShapeDataset(shapes_df[~shapes_df["val_set"]], shape_transforms["train"]), batch_size=64, shuffle=True)
val_loader = DataLoader(ShapeDataset(shapes_df[shapes_df["val_set"]], shape_transforms["val"]), batch_size=64, shuffle=False)

shape_net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT, progress=False)
shape_net.fc = nn.Linear(shape_net.fc.in_features, len(shape_classes))
for name, param in shape_net.named_parameters():
    if "bn" not in name and "fc" not in name:
        param.requires_grad = False

info_stats = torchinfo.summary(shape_net, input_size=(2, 3, 160, 160), verbose=0)
sample_imgs, _ = next(iter(val_loader))
forward_trace = tl.trace(shape_net, sample_imgs[:2].cpu())
print(f"Total params: {info_stats.total_params:,} | Trainable params: {info_stats.trainable_params:,} ({info_stats.trainable_params / info_stats.total_params:.2%})")
print(f"TorchLens traced {len(forward_trace.layer_labels)} elementary tensor operations from input to 6-class logits.")

Two complementary model-informatics libraries help demystify any unfamiliar architecture before training: `torchinfo.summary` audits parameter budgets and activation shapes, while JohnMark Taylor's [`torchlens`](https://github.com/johnmarktaylor91/torchlens) (`tl.trace`) records every elementary tensor operation — including inline additions and skip connections that module summaries miss. Now we run three quick epochs of transfer learning with a `OneCycleLR` learning-rate schedule:

In [ ]:
import copy

shape_net = shape_net.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([p for p in shape_net.parameters() if p.requires_grad], lr=3e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=2e-2, steps_per_epoch=len(train_loader), epochs=3)

history = []
best_acc, best_wts = 0.0, copy.deepcopy(shape_net.state_dict())
for epoch in range(1, 4):
    for phase, loader in (("train", train_loader), ("val", val_loader)):
        shape_net.train(phase == "train")
        correct, total, loss_sum = 0, 0, 0.0
        for imgs, labels_b in loader:
            imgs, labels_b = imgs.to(DEVICE), labels_b.to(DEVICE)
            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == "train"):
                logits = shape_net(imgs)
                loss = criterion(logits, labels_b)
                if phase == "train":
                    loss.backward()
                    optimizer.step()
                    scheduler.step()
            loss_sum += float(loss.item()) * imgs.size(0)
            correct += int((logits.argmax(dim=1) == labels_b).sum().item())
            total += imgs.size(0)
        acc = correct / total
        history.append({"epoch": epoch, "phase": phase, "loss": loss_sum / total, "accuracy": acc})
        if phase == "val" and acc > best_acc:
            best_acc, best_wts = acc, copy.deepcopy(shape_net.state_dict())

shape_net.load_state_dict(best_wts)
shape_net.eval()
print(f"Best held-out 6-way shape accuracy after 3 epochs: {best_acc:.1%} (chance = {1/6:.1%})")

## Modern Feature Visualization with `horama` (MACO)

Accuracy tells us *that* the readout separates spheres from cones, without telling us *which visual patterns* excite each classifying neuron. Classic pixel-space gradient ascent tends to synthesize high-frequency adversarial static unless heavily regularized. Building on work by [Thomas Fel and colleagues (2023)](https://github.com/fel-thomas/horama), the modern `horama` library solves this via **MACO (Magnitude-Constrained Optimization)**: fixing the natural-image $1/f$ Fourier magnitude spectrum and optimizing **only the Fourier phases**, while accumulating gradient magnitude into an opacity mask (`alpha`) that highlights the synthesized motif's informative region.

Below, we synthesize MACO feature visualizations for our trained `Sphere`, `Cone`, and `Torus` readout units:

In [ ]:
import horama

vis_steps = 96 if DEVICE.type == "cuda" else 44
target_units = ["Sphere", "Cone", "Torus"]
maco_renders = {}

for cname in target_units:
    cls_idx = shape_classes.index(cname)
    objective = lambda images, idx=cls_idx: torch.mean(shape_net(images)[:, idx])
    img_vis, alpha_vis = horama.maco(
        objective,
        total_steps=vis_steps,
        learning_rate=1.0,
        image_size=256,
        model_input_size=160,
        crops_per_iteration=4,
        device=str(DEVICE),
    )
    # Normalize RGB and blend with transparency mask against clean light background for inspection.
    rgb = img_vis.detach().cpu().permute(1, 2, 0).numpy()
    rgb = (rgb - rgb.min()) / (np.ptp(rgb) + 1e-8)
    alpha = alpha_vis.detach().cpu().permute(1, 2, 0).numpy()
    alpha = np.clip(alpha / (np.quantile(alpha, 0.99) + 1e-8), 0, 1)
    blended = alpha * rgb + (1.0 - alpha) * 0.96
    maco_renders[cname] = np.clip(blended, 0, 1)

shape_net = shape_net.cpu()

In [ ]:
# @title Show the figure code
fig, axes = plt.subplots(1, 3, figsize=(8.8, 3.1))
for ax, cname in zip(axes, target_units):
    ax.imshow(maco_renders[cname])
    ax.set_title(f"MACO phase synthesis: {cname}", fontsize=9.5)
    ax.set_axis_off()
plt.tight_layout()

Compare the synthesized prototypes above: the `Sphere` unit excites curved, concentrically shaded round contours, `Cone` excites tapered triangular silhouettes and apex angles, and `Torus` excites ring-like apertures with central hollows — confirming that our readout latches onto 3D geometric cues despite randomized surface textures.

> **Takeaway #5**
>
> Transfer learning reuses general-purpose perceptual features learned at scale; feature visualization and attribution audit whether a classifier relies on target geometry or incidental shortcuts like texture.

# Large Language Models and Mechanistic Interpretability

Vision models turn patches of pixels into spatial representations; **large language models (LLMs)** turn sequences of text into contextualized vector trajectories. Despite the mystique surrounding chat interfaces, the core engine inside a GPT-style autoregressive transformer follows a remarkably compact pipeline:

1. **Subword tokenization**: Raw text splits into common word pieces (`[Words][ be][come][ to][kens]`), each mapped to an integer lookup ID (`[24162][307]...`).
2. **Vectorization**: Each token ID retrieves an initial dense vector of floating-point coordinates.
3. **Hierarchical causal attention**: Across stacked transformer blocks, **attention heads** let each token look backward at earlier context tokens, routing syntactic and semantic cues into an evolving residual vector.
4. **Next-token readout on the sphere**: At the top of the stack, the final contextualized vector points along a trajectory in embedding space; taking dot products against the vocabulary matrix (followed by temperature-scaled `softmax`) yields probabilities over every candidate `[NEXT]` token. Repeat that loop one token at a time — and add post-training instruction tuning — and you have a modern assistant model.

Press **play** in Figure 6 below (backed by real GPT-2 tokenizations, attention weights, and 3D-projected residual trajectories), switch between prompts, rotate the 3D sphere, and slide **Temperature** to watch sharper versus flatter next-token distributions:

In [ ]:
# @title Show the interactive figure code
show("llm_explainer", data=widget_sample("llm_explainer"))

## Modern Hugging Face Finetuning with LoRA (`peft` + `Trainer`)

Because pretrained language transformers already encode rich lexical, syntactic, and pragmatic regularities, adapting one to a behavioral or text-analytic classification task requires neither training from scratch nor updating every parameter in the model. Modern [Hugging Face](https://huggingface.co/docs/transformers) workflows combine `AutoModelForSequenceClassification`, dynamic batch padding (`DataCollatorWithPadding`), and **Low-Rank Adaptation (`peft` LoRA)** — freezing 98%+ of the pretrained transformer weights while training tiny rank-$r$ adapter matrices injected into the attention projections.

Below, we load the Sarcasm Headlines dataset (`load_sarcasm`, $28,619$ headlines from *The Onion* vs. *HuffPost*), wrap `distilbert-base-uncased` in a rank-8 LoRA adapter, and finetune with Hugging Face's `Trainer`:

In [ ]:
from datasets import Dataset as HFDataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model

sarcasm_df = dl4hl.load_sarcasm()
print(f"Loaded {len(sarcasm_df):,} headlines ({sarcasm_df['label'].mean():.1%} sarcastic)")

# Use a larger slice when a GPU is active and a compact stratified subset on CPU so workshop cells stay brisk.
n_sub = 2400 if DEVICE.type == "cuda" else 800
sarc_sub = pd.concat([
    g.sample(n_sub // 2, random_state=2026) for _, g in sarcasm_df.groupby("label")
]).sample(frac=1.0, random_state=2026).reset_index(drop=True)

lm_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(lm_name)
hf_ds = HFDataset.from_pandas(sarc_sub[["sentence", "label"]])
hf_ds = hf_ds.map(lambda b: tokenizer(b["sentence"], truncation=True, max_length=48), batched=True, remove_columns=["sentence"])
hf_split = hf_ds.train_test_split(test_size=0.25, seed=2026)

base_lm = AutoModelForSequenceClassification.from_pretrained(lm_name, num_labels=2)
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_lin", "v_lin"], modules_to_save=["pre_classifier", "classifier"],
)
lora_lm = get_peft_model(base_lm, lora_cfg)
trainable_p, total_p = lora_lm.get_nb_trainable_parameters()
print(f"LoRA trainable parameters: {trainable_p:,} / {total_p:,} ({trainable_p / total_p:.2%})")

train_args = TrainingArguments(
    output_dir=".cache/sarcasm_lora",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-4,
    weight_decay=0.01,
    logging_strategy="no",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    seed=2026,
    dataloader_pin_memory=(DEVICE.type == "cuda"),
)
lm_trainer = Trainer(
    model=lora_lm,
    args=train_args,
    train_dataset=hf_split["train"],
    eval_dataset=hf_split["test"],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=lambda p: {"accuracy": float((np.argmax(p.predictions, axis=-1) == p.label_ids).mean())},
)
lm_trainer.train()
eval_metrics = lm_trainer.evaluate()
print(f"Held-out sarcasm classification accuracy: {eval_metrics['eval_accuracy']:.1%}")

> **Corpus shortcuts in headline classification**
>
> Because this benchmark contrasts satirical *Onion* headlines against straight *HuffPost* headlines, a classifier can exploit stylistic and topical register cues (such as *"area man"* or *"report:"*) alongside genuine pragmatic irony. Token attribution and sparse feature discovery below help inspect which cues actually drive predictions.

We can query our LoRA-adapted transformer on new headlines and compute gradient-times-embedding token attributions to see which subwords push the decision toward `sarcastic` versus `straight`:

In [ ]:
lora_lm.eval()
lm_dev = next(lora_lm.parameters()).device

def explain_headline(prompt: str) -> pd.DataFrame:
    enc = tokenizer(prompt, return_tensors="pt").to(lm_dev)
    embed_layer = lora_lm.get_input_embeddings()
    embeds = embed_layer(enc["input_ids"]).detach().requires_grad_(True)
    out = lora_lm(inputs_embeds=embeds, attention_mask=enc["attention_mask"])
    probs = torch.softmax(out.logits[0], dim=-1)
    sarc_logit = out.logits[0, 1] - out.logits[0, 0]
    sarc_logit.backward()
    # Signed gradient x input attribution per token toward 'sarcastic'
    scores = (embeds.grad[0] * embeds.detach()[0]).sum(dim=-1).cpu().numpy()
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][0].cpu().tolist())
    return pd.DataFrame({"token": toks, "attribution": scores, "p_sarcastic": float(probs[1].item())})

probe_a = explain_headline("5 simple ways to organize your family kitchen this weekend")
probe_b = explain_headline("local graduate student weeps on keyboard, again")
print(f"Probe A P(sarcastic) = {probe_a['p_sarcastic'].iloc[0]:.2f} | Probe B P(sarcastic) = {probe_b['p_sarcastic'].iloc[0]:.2f}")

In [ ]:
# @title Show the figure code
inner = probe_b[~probe_b["token"].isin(["[CLS]", "[SEP]"])].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8.4, 2.9))
bar_colors = [C["orange"] if v >= 0 else C["blue"] for v in inner["attribution"]]
ax.bar(range(len(inner)), inner["attribution"], color=bar_colors)
ax.axhline(0, color=C["ink"], lw=0.9)
ax.set_xticks(range(len(inner)))
ax.set_xticklabels(inner["token"], rotation=20)
ax.set(ylabel="signed attribution (\u2192 sarcastic)", title='Token-level attribution on "Local Graduate Student Weeps on Keyboard, Again"')
plt.tight_layout()

> **Takeaway #6**
>
> Language transformers convert subword tokens into contextualized vector trajectories that predict upcoming tokens; parameter-efficient finetuning adapts those trajectories to new tasks.

## Mechanistic Interpretability with Sparse Autoencoders (`overcomplete`)

Why can't we simply read off what a language model knows by inspecting one hidden neuron at a time? Because real-world concepts are **sparse** (any single sentence mentions only a tiny handful of topics out of tens of thousands), networks pack far more concepts than they have neurons into overlapping, nearly orthogonal directions — a phenomenon called **superposition**. Consequently, individual hidden coordinates are **polysemantic**, firing for mixtures of unrelated themes.

**Sparse Autoencoders (SAEs)** reverse that compression. Instead of squeezing activations through a narrow bottleneck, an SAE maps a $d$-dimensional hidden vector into a *wider* (**overcomplete**) dictionary of $M$ concept directions while enforcing strict sparsity (such as keeping only the **Top-$K$** active features per input) before reconstructing the original vector. Explore the superposition pentagon and real discovered DistilBERT SAE latents in Figure 7 below:

In [ ]:
# @title Show the interactive figure code
show("sae_superposition", data=widget_sample("sae_superposition"))

Using Thomas Fel's [`overcomplete`](https://github.com/fel-thomas/overcomplete) library, we can train a real `TopKSAE` (`48` dictionary concepts, `top_k=4` active per headline) directly on $3,000$ DistilBERT `[CLS]` activations from the sarcasm corpus in just a few seconds:

In [ ]:
from overcomplete.sae import TopKSAE, train_sae

sarc_bundle = dl4hl.load_sarcasm(embeddings=True)
X_cls = StandardScaler().fit_transform(sarc_bundle["embeddings"]).astype(np.float32)
sents_3k = sarc_bundle["sentences"]

torch.manual_seed(2026)
sae_model = TopKSAE(input_shape=X_cls.shape[1], nb_concepts=48, top_k=4, device=str(DEVICE))
sae_opt = torch.optim.Adam(sae_model.parameters(), lr=3e-3)
sae_loader = DataLoader(TensorDataset(torch.from_numpy(X_cls)), batch_size=128, shuffle=True)

def sae_mse_loss(x, x_hat, pre_codes, codes, dictionary):
    return torch.mean((x - x_hat) ** 2)

train_sae(sae_model, sae_loader, sae_mse_loss, sae_opt, nb_epochs=10, monitoring=0, device=str(DEVICE))
sae_model.eval()
with torch.no_grad():
    _, sparse_codes, X_recon = sae_model(torch.from_numpy(X_cls).to(DEVICE))
sparse_np = sparse_codes.cpu().numpy()
recon_r2 = 1.0 - np.sum((X_cls - X_recon.cpu().numpy()) ** 2) / np.sum(X_cls ** 2)
print(f"TopKSAE (k=4 of 48, 91.7% zeros per input) reconstructs {recon_r2:.1%} of dense activation variance.")

# Inspect the 3 highest-peaking dictionary latents with moderate firing frequency.
fire_freq = (sparse_np > 0).mean(axis=0)
good_latents = np.where((fire_freq > 0.04) & (fire_freq < 0.30))[0]
top_latents = good_latents[np.argsort(sparse_np[:, good_latents].max(axis=0))[::-1][:3]]
for lid in top_latents:
    best_rows = np.argsort(sparse_np[:, lid])[::-1][:3]
    examples = " | ".join(sents_3k[r][:62] + "..." for r in best_rows)
    print(f"Latent #{lid:02d} (fires {fire_freq[lid]:.0%}): {examples}")
sae_model = sae_model.cpu()

Without a single topic label, enforcing `top_k=4` sparsity decomposes dense 768-dimensional activations into coherent thematic directions — separating everyday domestic irony, electoral politics, and emergency/international news into distinct dictionary columns.

> **Takeaway #7**
>
> Individual hidden units often superimpose multiple unrelated concepts; sparse autoencoders expand dense activations into wider, sparsely active dictionaries whose features are easier to interpret.

# Multimodal Alignment, Zero-Shot Probing, and Relative Representations

Unimodal vision networks know what pixels go together; unimodal language models know what words go together. **Contrastive vision-language models** like **CLIP** ([Radford et al., 2021](https://arxiv.org/abs/2103.00020)) and **SigLIP** train an image encoder and a text encoder jointly across hundreds of millions of paired images and captions, pulling matched image–caption vectors together on a shared unit hypersphere while pushing mismatched pairs apart.

Once aligned, that shared geometry unlocks **zero-shot probing**: we can test visual hypotheses using natural-language prompts without training a single new weight.

## Zero-Shot Probing and the Case of Kiki–Bouba

To see both the power and the methodological traps of zero-shot probing, we turn to classic psycholinguistic sound symbolism: humans reliably match jagged, star-like shapes with sharp-consonant pseudowords (*kiki*, *kitaki*) and smooth, lobular shapes with rounded-vowel pseudowords (*bouba*, *balouba*). Has a multimodal model trained only on internet images and written text absorbed that cross-modal iconicity?

We load Dr. Sammy Floyd's parametrically generated Kiki–Bouba stimuli (`load_kiki_bouba`, $100$ shapes) alongside `OpenCLIP` (`ViT-B-32-quickgelu`) and run the trinary forced-choice probe (`kitaki`, `balouba`, plus phonetic distractor `leam`), comparing raw pseudoword prompts against explicit geometric descriptions:

In [ ]:
import open_clip

kibo = dl4hl.load_kiki_bouba()
kibo_df = kibo["df"]

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32-quickgelu", pretrained="openai")
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32-quickgelu")
clip_model = clip_model.to(DEVICE).eval()

kibo_imgs = torch.stack([clip_preprocess(Image.open(p)) for p in kibo_df["image_path"]]).to(DEVICE)
with torch.no_grad():
    img_vecs = clip_model.encode_image(kibo_imgs)
    img_vecs = img_vecs / img_vecs.norm(dim=-1, keepdim=True)

def zeroshot_probe(prompts: list[str], choice_names: list[str]) -> pd.DataFrame:
    with torch.no_grad():
        txt_vecs = clip_model.encode_text(clip_tokenizer(prompts).to(DEVICE))
        txt_vecs = txt_vecs / txt_vecs.norm(dim=-1, keepdim=True)
        probs = (100.0 * img_vecs @ txt_vecs.T).softmax(dim=-1).cpu().numpy()
    preds = [choice_names[i] for i in probs.argmax(axis=1)]
    return pd.DataFrame({"actual": kibo_df["iconic_label"], "predicted": preds})

iconic_choices = ["kitaki", "balouba", "leam"]
bare_res = zeroshot_probe(iconic_choices, iconic_choices)
framed_res = zeroshot_probe([f"a picture of a {c}" for c in iconic_choices], iconic_choices)
print(f"Bare-token prompt accuracy: {(bare_res['actual'] == bare_res['predicted']).mean():.1%}")
print(f"Framed prompt ('a picture of a ...') accuracy: {(framed_res['actual'] == framed_res['predicted']).mean():.1%}")
pd.crosstab(bare_res["actual"], bare_res["predicted"], rownames=["Actual"], colnames=["Predicted (bare token)"])

In [ ]:
# @title Show the figure code
sample_kb = (
    [kibo_df.query("shape_family == 'bouba'")["image_path"].iloc[i] for i in range(3)]
    + [kibo_df.query("shape_family == 'kiki'")["image_path"].iloc[i] for i in range(3)]
)
fig, axes = plt.subplots(1, 6, figsize=(9.2, 2.1))
for idx, (ax, p) in enumerate(zip(axes, sample_kb)):
    ax.imshow(Image.open(p))
    ax.set_title(f"{'bouba' if idx < 3 else 'kiki'} #{(idx % 3) + 1}", fontsize=9)
    ax.set_axis_off()
fig.suptitle("Parametric Kiki–Bouba stimuli (Floyd et al.): rounded lobes vs. sharp angular spikes", fontsize=10)
plt.tight_layout()

Compare those two prompt framings and the confusion table above: changing nothing in the visual encoder — only wrapping bare pseudowords (`"kitaki"`) in a natural caption template (`"a picture of a kitaki"`) — shifts zero-shot accuracy substantially, while the confusion matrix exposes any asymmetric prior pull toward one unfamiliar spelling or distractor token. Always inspect the full confusion matrix alongside zero-shot accuracy, and average across multiple **prompt templates** (`"a drawing of a {}"`, `"a silhouette of a {}"`, `"a black-and-white shape of a {}"`) to stabilize linguistic priors.

## Relative Representations and Anchor-Query Classifiers

Zero-shot classification uses text prompts as class prototypes, yet we can push that shared geometry much further. Following the **relative representations** framework ([Moschella et al., 2022](https://arxiv.org/abs/2209.15430)), suppose we specify $K$ interpretable natural-language **anchor queries** — *"Is spiky or jagged"*, *"Is smooth and rounded"*, *"Shows an outdoor entrance"*, *"Shows a cooking hearth"* — and represent every image solely by its $K$ cosine similarities to those anchors. Instead of fitting an opaque classifier over 512 unnamed embedding coordinates, we fit a regularized linear model directly over the **query-based subspace**, turning every regression weight into a readable **coefficient over queries**.

Toggle individual anchor queries in or out of the subspace and rotate the 3D sphere in Figure 8 below:

In [ ]:
# @title Show the interactive figure code
show("relative_reps", data=widget_sample("relative_reps"))

In code, projecting our 100 Kiki–Bouba embeddings onto five natural-language shape descriptors yields a 5-column design matrix whose cross-validated logistic regression coefficients recover the exact perceptual contrast between spiky and rounded forms:

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

anchor_queries = [
    "a spiky shape with sharp jagged points",
    "a smooth rounded shape with soft lobes",
    "an angular starburst or crystalline polygon",
    "a curvy amoeba or cloud silhouette",
    "a symmetrical geometric emblem",
]
short_queries = ["spiky / jagged", "smooth / rounded", "angular starburst", "curvy cloud", "symmetrical"]

with torch.no_grad():
    q_vecs = clip_model.encode_text(clip_tokenizer(anchor_queries).to(DEVICE))
    q_vecs = q_vecs / q_vecs.norm(dim=-1, keepdim=True)
    query_coords = (img_vecs @ q_vecs.T).cpu().numpy()

y_kiki = (kibo_df["shape_family"] == "kiki").astype(int).to_numpy()
query_pipe = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=10, cv=5))
query_cv_acc = cross_val_score(query_pipe, query_coords, y_kiki, cv=5).mean()
query_pipe.fit(query_coords, y_kiki)
query_coefs = pd.Series(query_pipe.named_steps["logisticregressioncv"].coef_[0], index=short_queries)
print(f"5-fold accuracy over 5-query subspace: {query_cv_acc:.1%}")
query_coefs.round(2)

In [ ]:
# @title Show the figure code
fig, ax = plt.subplots(figsize=(7.6, 2.9))
cols_q = [C["orange"] if v >= 0 else C["blue"] for v in query_coefs.values]
ax.barh(query_coefs.index, query_coefs.values, color=cols_q)
ax.axvline(0, color=C["ink"], ls="--", lw=1.0)
ax.set(xlabel="standardized logistic weight (\u2190 bouba  |  kiki \u2192)", title="Interpretable classifier coefficients over natural-language anchor queries")
plt.tight_layout()

> **Takeaway #8**
>
> Contrastive multimodal models align images and text in a shared geometry, enabling zero-shot probing and interpretable classifiers built from similarity to natural-language anchor queries.

# Encoding, Decoding, and the Neuroconnectionist Model Zoo

Up to now we have mostly read from the *top* of pretrained networks. For cognitive scientists, developmentalists, and aesthetic researchers, the richest signal often lives *inside* the hierarchy: **where along a model's layers do low-level visual statistics give way to abstract categories or subjective human judgments?**

Two complementary linear-probing paradigms turn any frozen network into an empirical measurement instrument:

- **Encoding models** map a model layer's features onto continuous human responses — such as normative ratings of **beauty**, **valence**, or **arousal** — using cross-validated regularized regression (`RidgeCV`) to measure how strongly that layer predicts human judgments on unseen stimuli.
- **Decoding models** use the exact same extracted layer features in a regularized linear classifier (`RidgeClassifierCV`) to test whether an experimental variable (such as 3D object identity across randomized textures) is linearly decodable at that stage of processing.

## Layerwise Aesthetics Encoding and Shape Decoding with `DeepDive`

To run both paradigms cleanly, we load [`DeepDive`](https://github.com/ColinConwell/DeepDive) (`dl4hl.ensure_deepdive()`), a PyTorch feature-extraction and dimensionality-reduction toolkit that hooks arbitrary intermediate layers during a single batched forward pass (`get_all_feature_maps`) while preserving exact stimulus order (`shuffle=False`), then compresses wide intermediate tensors down to tractable dimensions via Johnson–Lindenstrauss **Sparse Random Projection** (`get_feature_map_srps`).

Scrub or press **play** across Vision Transformer depth in Figure 9 below to inspect how aesthetic encoding ($r$) and 3D shape decoding climb from the patch stem to the final normalized representation, and toggle the right panel between the **Beauty scatter** and the **Layer RDM**:

In [ ]:
# @title Show the interactive figure code
show("layerwise_readout", data=widget_sample("layerwise_readout"))

Below, we execute this exact `DeepDive` pipeline end to end on the 900-image **Open Affective Standardized Image Set (OASIS; [Kurdi, Lozano, & Banaji, 2017](https://doi.org/10.3758/s13428-016-0715-3), `load_oasis`)** — reproducing the core aesthetics regression from *Beauty Ex Machina* ([Conwell et al., 2021](https://osf.io/preprints/psyarxiv/5wg4s)) — followed by layerwise 3D object decoding on `SimpleShapes`:

In [ ]:
from scipy.stats import pearsonr
from sklearn.linear_model import RidgeClassifierCV

dl4hl.ensure_deepdive()
from feature_extraction import StimulusSet, get_all_feature_maps, get_empty_feature_maps
from feature_reduction import get_feature_map_srps

oasis = dl4hl.load_oasis()
oasis_df = oasis["df"]
# On CPU runtimes, use a stratified 360-image subset so multi-layer ViT extraction finishes in seconds; on GPU use all 900.
if DEVICE.type == "cuda":
    oasis_eval = oasis_df.reset_index(drop=True)
else:
    oasis_eval = (
        pd.concat([g.sample(min(90, len(g)), random_state=2026) for _, g in oasis_df.groupby("image_type")])
        .sort_values("image_name")
        .reset_index(drop=True)
    )

# IMPORTANT: never shuffle a feature-extraction DataLoader, or stimulus rows decouple from rating rows!
oasis_loader = DataLoader(StimulusSet(oasis_eval["image_path"], clip_preprocess), batch_size=64, shuffle=False)
vit_backbone = clip_model.visual.eval()

all_vit_layers = get_empty_feature_maps(vit_backbone, inputs=next(iter(oasis_loader))[:3], names_only=True)
sampled_layers = ["Conv2d-1", "ResidualAttentionBlock-2", "ResidualAttentionBlock-5", "ResidualAttentionBlock-8", "ResidualAttentionBlock-11", "VisionTransformer-1"]
print(f"Hooked {len(sampled_layers)} of {len(all_vit_layers)} ViT layers: {sampled_layers}")

raw_oasis_maps = get_all_feature_maps(vit_backbone, oasis_loader, layers_to_retain=sampled_layers, use_tqdm=False)
oasis_maps = get_feature_map_srps(raw_oasis_maps, n_projections=512, delete_originals=True, upsampling=False, seed=0)

alpha_grid = np.logspace(-1, 5, 19)
encoding_rows = []
for depth_idx, lname in enumerate(sampled_layers):
    X_layer = StandardScaler().fit_transform(oasis_maps[lname])
    for rating_col in ("beauty", "valence", "arousal"):
        y_rating = oasis_eval[rating_col].to_numpy(dtype=float)
        # scoring='explained_variance' stores out-of-fold leave-one-out predictions in cv_results_.
        ridge = RidgeCV(alphas=alpha_grid, scoring="explained_variance", store_cv_results=True).fit(X_layer, y_rating)
        best_col = int(np.argmin(np.abs(alpha_grid - ridge.alpha_)))
        y_oof = ridge.cv_results_[:, best_col]
        encoding_rows.append({
            "depth": depth_idx,
            "layer": lname,
            "dimension": rating_col,
            "pearson_r": float(pearsonr(y_rating, y_oof).statistic),
            "alpha": float(ridge.alpha_),
        })

encoding_results = pd.DataFrame(encoding_rows)
encoding_results.pivot(index="layer", columns="dimension", values="pearson_r").reindex(sampled_layers).round(3)

Now apply the identical `DeepDive` feature-extraction and `SparseRandomProjection` workflow to decode 3D shape identity across those same six Vision Transformer stages on `SimpleShapes`:

In [ ]:
shapes_sub = shapes_df[shapes_df["image_index"] <= (120 if DEVICE.type == "cuda" else 60)].reset_index(drop=True)
shapes_loader = DataLoader(StimulusSet(shapes_sub["image_path"], clip_preprocess), batch_size=64, shuffle=False)

raw_shape_maps = get_all_feature_maps(vit_backbone, shapes_loader, layers_to_retain=sampled_layers, use_tqdm=False)
shape_maps = get_feature_map_srps(raw_shape_maps, n_projections=512, delete_originals=True, upsampling=False, seed=0)

y_shape_codes, shape_names = pd.factorize(shapes_sub["label"], sort=True)
is_val_shape = shapes_sub["val_set"].to_numpy()

decoding_rows = []
for depth_idx, lname in enumerate(sampled_layers):
    X_sh = StandardScaler().fit_transform(shape_maps[lname])
    clf = RidgeClassifierCV(alphas=alpha_grid).fit(X_sh[~is_val_shape], y_shape_codes[~is_val_shape])
    preds_val = clf.predict(X_sh[is_val_shape])
    for code_i, cname in enumerate(shape_names):
        mask_c = y_shape_codes[is_val_shape] == code_i
        decoding_rows.append({
            "depth": depth_idx,
            "layer": lname,
            "category": cname,
            "accuracy": float((preds_val[mask_c] == code_i).mean()),
        })

decoding_results = pd.DataFrame(decoding_rows)
clip_model = clip_model.cpu()
print("Mean held-out 6-way shape decoding at patch stem vs. final layer:",
      round(decoding_results.query("depth == 0")["accuracy"].mean(), 3), "->",
      round(decoding_results.query("depth == 5")["accuracy"].mean(), 3))

In [ ]:
# @title Show the figure code
short_stage = ["Stem", "Block 2", "Block 5", "Block 8", "Block 11", "Final"]
fig, axes = plt.subplots(1, 2, figsize=(9.8, 3.6))

dim_colors = {"beauty": C["blue"], "valence": C["green"], "arousal": C["orange"]}
for dim, grp in encoding_results.groupby("dimension"):
    grp_s = grp.sort_values("depth")
    axes[0].plot(grp_s["depth"], grp_s["pearson_r"], "o-", lw=2.2, color=dim_colors[dim], label=dim)
axes[0].set_xticks(range(len(short_stage)))
axes[0].set_xticklabels(short_stage, rotation=20)
axes[0].set(ylim=(-0.05, 1.0), ylabel="LOOCV Pearson r", title="OASIS aesthetics encoding across ViT depth")
axes[0].legend()

for idx, (cat, grp) in enumerate(decoding_results.groupby("category")):
    grp_s = grp.sort_values("depth")
    axes[1].plot(grp_s["depth"], grp_s["accuracy"], "o-", lw=1.3, alpha=0.65, color=dl4hl.PALETTE[idx], label=cat)
mean_dec = decoding_results.groupby("depth")["accuracy"].mean()
axes[1].plot(mean_dec.index, mean_dec.values, "o-", lw=2.8, color=C["ink"], label="Mean (all 6)")
axes[1].set_xticks(range(len(short_stage)))
axes[1].set_xticklabels(short_stage, rotation=20)
axes[1].set(ylim=(0.0, 1.02), ylabel="held-out decoding accuracy", title="SimpleShapes 3D object decoding across ViT depth")
axes[1].legend(fontsize=7.5, ncol=2)
plt.tight_layout()

Look at the parallel rise across both panels: in the patch stem (`Conv2d-1`), local pixel patches carry almost zero linear signal for subjective beauty, valence, arousal, or texture-invariant 3D shape. As self-attention integrates global context across blocks, both human affective-aesthetic structure and 3D shape invariance emerge progressively — peaking in the language-aligned output layer (`r > 0.80` for aesthetic ratings and `> 85%` for 6-way shape decoding).

> **Going faster: DeepJuice and Sim-Acela**
>
> Benchmarking dozens or hundreds of large models across layers quickly bottlenecks CPU memory and scikit-learn solvers. [`DeepJuice`](https://github.com/ColinConwell/DeepJuice) upgrades `DeepDive` with GPU-batched sparse random projections and closed-form PyTorch `TorchRidgeGCV` solvers capable of fitting thousands of outcomes per second, while [`Sim-Acela`](https://github.com/ColinConwell/Sim-Acela) accelerates multi-metric representational alignment (RSA, CKA, Procrustes, and encoding-RSA) on GPU.

> **Takeaway #9**
>
> Linear encoding and decoding probes treat frozen network layers as candidate feature spaces, measuring where along a processing hierarchy perceptual, affective, or categorical information becomes linearly accessible.

## Types of Deep Learning Models: The Neuroconnectionist Trifecta

With supervised classifiers, autoencoders, autoregressive LLMs, and contrastive vision-language models now in hand, how should a human-sciences researcher organize the sprawling zoo of deep learning architectures and training paradigms? Following the **neuroconnectionist** framework ([Doerig et al., 2023](https://doi.org/10.1038/s41583-023-00705-w); [Conwell et al., 2024](https://doi.org/10.1038/s41467-024-53147-y)), every learned representation reflects three distinct pressures:

1. **Sensory Diet (Input Environment)**: What statistical stream of experience does the learner observe — object-centric snapshots (`ImageNet`), navigable scenes (`Places365`), faces (`VGGFace2`), domestic environments (`Dollar Street`), or ungrounded text corpora?
2. **Computational Architecture**: What structural circuit processes that stream — local 2D spatial convolutions (`ResNet`, `ConvNeXt`), global token-to-token attention (`ViT`, `GPT`), or recurrent state spaces?
3. **Computational Objective (Task / Learning Signal)**: What loss function tells the weights how to change?
   - **Category-supervised learning** supplies explicit human category labels (*"golden retriever"* vs. *"Finnish spitz"*). Powerful for engineering, it faces Quine's classic *Gavagai* (**inscrutability of reference**) problem as a model of early human development: infants do not experience the world stamped with million-item answer keys.
   - **Unimodal self-supervised learning** discards human labels and mines supervision from the structure of sensory data itself: **autoencoding** (compress and reconstruct), spatial **pretext puzzles** (unscramble **Jigsaw** patches or predict $0^\circ/90^\circ/180^\circ/270^\circ$ rotations in **RotationNet**), **DeepCluster** (iterative latent *k*-means), and **instance-contrastive learning** (`SimCLR`, `BarlowTwins`, `DINO`, `IPCL`), which pulls augmented views (crops, rotations, color shifts) of the *same* instance together on the hypersphere while pushing distinct instances apart — turning data augmentations into explicit, testable hypotheses about perceptual invariance.
   - **Multimodal alignment (`CLIP`, `SigLIP`) and sequence prediction (`BERT`, `GPT`)** align visual encoders to linguistic captions or predict masked/upcoming tokens across vast corpora.

Crucially, you do not need an industrial GPU cluster to test how these three pressures shape representations. Because modern open-source repositories (`Open-IPCL`, `timm`, `Taskonomy`, `VISSL`, `OpenCLIP`) contain models trained under systematic combinations of diet, architecture, and objective, we can run **opportunistic experiments**: varying **one** factor of the trifecta while holding the other two constant (such as comparing identical self-supervised `AlexNet` models reared on objects versus scenes versus faces in [Konkle & Alvarez, 2022](https://doi.org/10.1016/j.neuron.2021.12.008)).

Switch **Objective** and press **play** on the left of Figure 10 below to see how five canonical learning signals sculpt the latent hypersphere, and toggle **Vary 1 factor** on the right to design controlled open-source model comparisons:

In [ ]:
# @title Show the interactive figure code
show("trifecta_explorer")

> **Takeaway #10**
>
> Every deep network reflects three interacting pressures — sensory input, architecture, and training objective; comparing open-source models that vary one pressure while holding the others constant turns model zoos into controlled scientific experiments.

# Wrapping Up

## The Ten Takeaways

1. Stacking linear layers without nonlinear activations collapses to a single linear transformation; nonlinear hinges let a network approximate curved functions by combining local bends.
2. A single linear boundary cannot separate interaction patterns like XOR, where an input's meaning reverses depending on context; hidden nonlinear layers re-represent the inputs so a linear boundary at the end suffices.
3. Deep networks untangle complex data manifolds progressively across layers, applying a sequence of simple warps and folds until target categories or quantities become linearly readable.
4. Tabular neural networks pair PyTorch modules with familiar scikit-learn pipelines and grouped cross-validation, though gradient-boosted trees remain essential baselines on small structured tables.
5. Transfer learning reuses general-purpose perceptual features learned at scale; feature visualization and attribution audit whether a classifier relies on target geometry or incidental shortcuts like texture.
6. Language transformers convert subword tokens into contextualized vector trajectories that predict upcoming tokens; parameter-efficient finetuning adapts those trajectories to new tasks.
7. Individual hidden units often superimpose multiple unrelated concepts; sparse autoencoders expand dense activations into wider, sparsely active dictionaries whose features are easier to interpret.
8. Contrastive multimodal models align images and text in a shared geometry, enabling zero-shot probing and interpretable classifiers built from similarity to natural-language anchor queries.
9. Linear encoding and decoding probes treat frozen network layers as candidate feature spaces, measuring where along a processing hierarchy perceptual, affective, or categorical information becomes linearly accessible.
10. Every deep network reflects three interacting pressures — sensory input, architecture, and training objective; comparing open-source models that vary one pressure while holding the others constant turns model zoos into controlled scientific experiments.

## The Whole Toolkit on One Page

The following template captures the core linear-probing and encoding pattern used throughout cognitive science and empirical AI research — extract ordered representations from a frozen pretrained backbone, compress if wide, and evaluate a regularized linear readout across held-out groups or stimuli:

```python
import torch
from torch.utils.data import DataLoader
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.random_projection import SparseRandomProjection
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold, cross_val_score

# 1. Extract frozen representations in deterministic stimulus order (shuffle=False!).
backbone.eval()
features = []
with torch.no_grad():
    for batch in DataLoader(stimulus_dataset, batch_size=64, shuffle=False):
        features.append(backbone(batch.to(device)).cpu().numpy())
X = np.concatenate(features, axis=0)

# 2. Compress wide feature maps if needed and score with grouped cross-validation.
probe = make_pipeline(SparseRandomProjection(512, random_state=0), StandardScaler(), RidgeCV())
scores = cross_val_score(probe, X, y, groups=sampling_units, cv=GroupKFold(5))
print(f"Mean held-out R² across 5 group folds: {scores.mean():.3f}")
```

| Research goal | Start with | Upgrade to |
|---|---|---|
| fit nonlinear tabular interactions | `HistGradientBoostingRegressor` + `skorch` MLP | `lightning` multi-task modules, `TabPFN` |
| classify images with small $N$ | `torchvision` / `timm` frozen backbone + linear head | full finetuning with `OneCycleLR` + data augmentation |
| inspect model internals & features | `torchinfo`, `torchlens.trace`, `horama.maco` | layerwise `overcomplete` concept dictionaries (Craft / NMF) |
| adapt a language transformer | Hugging Face `Trainer` + `peft` LoRA (`DistilBERT`, `SmolLM2`) | instruction-tuned causal LLMs + `overcomplete.TopKSAE` |
| probe visual concepts with text | `open_clip` zero-shot prompt-template ensembles | relative representations (`query_coords` + `LogisticRegressionCV`) |
| map deep layers to human ratings or brain data | `DeepDive` (`get_all_feature_maps` + `SRP` + `RidgeCV`) | [`DeepJuice`](https://github.com/ColinConwell/DeepJuice) + [`Sim-Acela`](https://github.com/ColinConwell/Sim-Acela) |

## Glossary

- **Activation function**: Nonlinear elementwise function (such as ReLU $\max(0, z)$ or GELU) applied after linear layers so stacked layers do not collapse into a single matrix multiply.

- **Attention head**: Transformer sub-circuit that computes context-dependent weights between token positions, routing relevant information from earlier tokens into the current representation.

- **Backpropagation**: Application of the calculus chain rule backward through a computation graph so every learnable weight receives the gradient of the loss with respect to itself.

- **Contrastive learning**: Self-supervised objective that pulls representations of paired or augmented views of the same instance together while pushing distinct instances apart on a hypersphere.

- **Disentanglement**: Progressive geometric unknotting across layers that separates intertwined data manifolds into coordinates where target variables are linearly readable.

- **Encoding / decoding probe**: Fitting a regularized linear regression (`encoding` continuous responses) or linear classifier (`decoding` discrete categories) on frozen intermediate activations of a network.

- **LoRA (Low-Rank Adaptation)**: Parameter-efficient finetuning technique that freezes pretrained weight matrices and learns small low-rank adapter updates ($B A$) using a tiny fraction of parameters.

- **MACO (Magnitude-Constrained Optimization)**: Feature-visualization technique (`horama`) that optimizes Fourier phases while locking the natural-image magnitude spectrum to synthesize clean neuron prototypes.

- **Polysemanticity / superposition**: Packing more sparse concepts than neurons into overlapping directions, causing single hidden units to respond to multiple unrelated concepts.

- **Relative representation**: Representing items by their cosine similarities to a set of anchor points (such as natural-language queries in CLIP) rather than raw coordinate axes.

- **Sparse Autoencoder (SAE)**: Neural dictionary model that maps dense activations into a wider latent layer with a strict sparsity constraint (such as `TopK`) to recover monosemantic features.

- **Sparse Random Projection (SRP)**: Fast, data-independent dimensionality reduction grounded in the Johnson–Lindenstrauss lemma that approximately preserves pairwise Euclidean distances.

- **Token**: Integer-indexed word or subword piece forming the elementary input and output vocabulary unit of a language transformer.

- **Transfer learning**: Reusing representations learned on a large upstream corpus or image set via frozen linear probing or finetuning on a downstream task.

## Data Sources

All datasets used in this tutorial are publicly available for research and educational use. Please cite the primary publications when using them in research.

| Dataset | Domain | Primary source |
|---|---|---|
| ShapeCentral (`load_simple_shapes`) | 3D perception, computer vision | Conwell ([ShapeCentral](https://github.com/ColinConwell/ShapeCentral)), Blender parametric 3D shape & texture benchmark |
| Open Affective Standardized Image Set (`load_oasis`) | empirical aesthetics, affective science | Kurdi, Lozano, & Banaji (2017), *Behavior Research Methods* ([OASIS](https://www.benedekkurdi.com/#oasis)); Conwell et al. (2021), *Beauty Ex Machina* |
| Parametric Kiki–Bouba (`load_kiki_bouba`) | psycholinguistics, sound symbolism | Floyd et al., parametrically generated spiky and rounded sound-symbolism stimuli |
| News Headlines for Sarcasm Detection (`load_sarcasm`) | computational pragmatics, NLP | Misra & Arora (2023), *AI Open* (*The Onion* & *HuffPost* headline corpus) |
| Stanford Wordbank MB-CDI (`load_wordbank`) | child development, language acquisition | Frank et al. (2017, 2021), [Wordbank](https://wordbank.stanford.edu) repository |
| IHDP longitudinal cohort (`load_perinatal_dev`) | developmental & clinical psychology | Brooks-Gunn et al. (1992), Infant Health and Development Program |
| World Bank LSMS household survey (`load_household_survey`) | anthropology, health economics | World Bank Living Standards Measurement Study; Cameron & Trivedi (2005) |
| 13-Dimensions of Music Emotion (`load_music_emotions`) | music therapy, affective science | Cowen et al. (2020), *PNAS* |
| Gapminder Dollar Street (`load_dollar_street`) | visual anthropology | [Gapminder Dollar Street](https://www.gapminder.org/dollar-street) (CC BY 4.0) |

Precomputed artifacts and the `dl4hl` helper package are hosted on the [`colinconwell/dl4hl`](https://huggingface.co/datasets/colinconwell/dl4hl) and [`colinconwell/ml4hl`](https://huggingface.co/datasets/colinconwell/ml4hl) Hugging Face dataset repositories and can be rebuilt from source via `scripts/build_artifacts.py`.